# Stage 3. Conjunctiva Segmentation

Notebook ini melatih U-Net pada foto mata utuh Eyes-Defy untuk memprediksi mask konjungtiva palpebral, lalu mengevaluasi kualitas segmentasi memakai Dice dan IoU serta menampilkan contoh overlay. Segmentasi ini menjadi jembatan dari citra mentah menuju region of interest pada mode capture di lapangan.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import torch
from PIL import Image

from configs import paths
from src import data, segmentation

manifest = data.assign_stratified_split(data.build_manifest(save=False))
print("device", "cuda" if torch.cuda.is_available() else "cpu")

## Segmentation Datasets

Hanya baris Eyes-Defy yang dipakai karena memuat foto mata utuh dan mask palpebral. Foto dimuat dengan koreksi orientasi EXIF agar sejajar dengan mask.

In [ ]:
train_ds, val_ds = segmentation.build_datasets(manifest, size=320)
print("train", len(train_ds), "val", len(val_ds))

## Train U-Net

Melatih U-Net dengan encoder ResNet34 pralatih ImageNet memakai kombinasi Dice loss dan binary cross entropy. Checkpoint terbaik disimpan berdasarkan Dice validasi.

In [ ]:
result = segmentation.train_unet(manifest, size=320, epochs=25, batch_size=8)
print("best val dice", round(result["metrics"]["best_val_dice"], 4))
print("final val iou", round(result["metrics"]["final_val_iou"], 4))

## Training Curves

In [ ]:
history = result["history"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_dice"], label="Dice")
axes[1].plot(history["val_iou"], label="IoU")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## Prediction Overlays

Overlay membandingkan citra, mask acuan, dan prediksi model pada data validasi.

In [ ]:
overlay_path = segmentation.save_overlays(result["model"], val_ds, count=4)
print("overlays saved to", overlay_path)
plt.figure(figsize=(9, 12))
plt.imshow(Image.open(overlay_path))
plt.axis("off")
plt.show()